# Workshop Notebook 1: Building an End-to-End ML Pipeline for Streamflow Prediction

**CIROH Developer's Conference 2026 | Foundations of Machine Learning**

---

## Goals

Build complete machine learning pipeline that predicts **daily streamflow** at 10 USGS-gauged basins in the southeastern United States, driven by daily climate forcings from the [CAMELS dataset](https://ral.ucar.edu/solutions/products/camels).

## Workflow overview

| # | Step | Key concepts |
|---|------|-------------|
| 1 | Explore the data | CAMELS forcings, streamflow, basin attributes |
| 2 | Preprocess | Normalization, temporal splits, sliding-window sequences |
| 3 | Build the model | `nn.Module`, LSTM architecture, parameters |
| 4 | Train | Loss function, optimizer, backpropagation, learning curves |
| 5 | Evaluate | Nash-Sutcliffe Efficiency (NSE), hydrograph plots |
| 6 | Diagnose | Overfitting, underfitting, hyperparameter tuning |

## The prediction task

Given a **365-day sequence** of daily climate variables, predict the streamflow at each of those 365 days:

```
Input x : (365 days) × [prcp, tmean, pet, dayl, srad, vp]
Output y : (365 days) × streamflow  (ft³/s)
```

<!-- This is a **sequence-to-sequence regression** problem — a natural fit for an LSTM. -->


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cpu")  # CPU-only for this workshop
print(f"Using device: {device}")

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "data"
sys.path.insert(0, str(ROOT))

print(f"PyTorch version: {torch.__version__}")


---
## 1. Exploring the CAMELS Dataset

The **CAMELS** (Catchment Attributes and Meteorology for Large-sample Studies) dataset provides long-record hydrometeorological time series for hundreds of US watersheds. We work with a curated **10-basin subset** from the southeastern US.

**Data dimensions:**
- **12,418 daily timesteps** — 1980-10-01 through 2014-09-30
- **6 forcing variables** - climate inputs that drive streamflow
- **1 target variable** — observed streamflow (ft³/s)
- **35 static basin attributes** — physical and geologic characteristics

| Forcing variable | Description |
|-----------------|-------------|
| `prcp`  | Precipitation (mm/day) |
| `tmean` | Mean air temperature (°C) |
| `pet`   | Potential evapotranspiration (mm/day) |
| `dayl`  | Day length (seconds) |
| `srad`  | Solar radiation (W/m²) |
| `vp`    | Vapor pressure (Pa) |


In [ ]:
from camels_loader import CamelsSubsetLoader, FORCING_NAMES, ATTRIBUTE_NAMES

loader = CamelsSubsetLoader(
    pickle_path=str(DATA_DIR / "camels_daymetv2"),
    gage_id_path=str(DATA_DIR / "gage_id.npy"),
)

print(loader)
print(f"\nForcing variables ({len(FORCING_NAMES)}): {FORCING_NAMES}")
print(f"Static attributes ({len(ATTRIBUTE_NAMES)}): {ATTRIBUTE_NAMES[:5]} ... (first 5 of {len(ATTRIBUTE_NAMES)})")
print(f"\nForcings array shape: {loader.forcings.shape} (time, basins, vars)")
print(f"Target array shape: {loader.target.shape} (time, basins, 1)")
print(f"Attributes shape: {loader.attributes.shape} (basins, attrs)")


In [ ]:
# Basin summary table
df_meta = pd.DataFrame({
    "Gage ID": loader.gage_ids,
    "Mean Q (ft³/s)": np.nanmean(loader.target[:, :, 0], axis=0).round(1),
    "Mean Precip (mm/d)": loader.forcings[:, :, FORCING_NAMES.index("prcp")].mean(0).round(2),
    "Basin Area (km²)": loader.attributes[:, ATTRIBUTE_NAMES.index("area_gages2")].round(0).astype(int),
    "Forest Cover (%)": (loader.attributes[:, ATTRIBUTE_NAMES.index("frac_forest")] * 100).round(1),
    "Mean Elevation (m)": loader.attributes[:, ATTRIBUTE_NAMES.index("elev_mean")].round(0).astype(int),
}).set_index("Gage ID")

df_meta


In [ ]:
# Visualize climate forcings and streamflow for one basin
# Try changing basin_idx (0–9) to explore different basins

basin_idx = 0
dates = loader.dates

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
fig.suptitle(
    f"Basin {loader.gage_ids[basin_idx]} — Daily Climate Forcings & Streamflow",
    fontsize=13, fontweight="bold"
)

axes[0].bar(dates, loader.forcings[:, basin_idx, FORCING_NAMES.index("prcp")],
            color="steelblue", width=1, alpha=0.75, label="Precipitation")
axes[0].set_ylabel("Precip (mm/day)")
axes[0].legend(loc="upper right", fontsize=8)

axes[1].plot(dates, loader.forcings[:, basin_idx, FORCING_NAMES.index("tmean")],
             color="darkorange", linewidth=0.5, label="Mean Temperature")
axes[1].axhline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.4)
axes[1].set_ylabel("Temp (°C)")
axes[1].legend(loc="upper right", fontsize=8)

sf = loader.target[:, basin_idx, 0]
axes[2].fill_between(dates, sf, alpha=0.35, color="navy", linewidth=0)
axes[2].plot(dates, sf, color="navy", linewidth=0.4, label="Streamflow")
axes[2].set_ylabel("Streamflow (ft³/s)")
axes[2].set_ylim(bottom=0)
axes[2].legend(loc="upper right", fontsize=8)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()


**Key observations to discuss:**
- Streamflow lags behind precipitation — water takes time to travel through the watershed
- Seasonal patterns in temperature drive snowmelt and evapotranspiration signals
- There are gaps (NaN) in the streamflow record — we'll handle these in preprocessing



---
## 2. Data Preprocessing

Raw data must be transformed before it can be fed into a neural network.

| Step | Why it matters |
|------|---------------|
| **Temporal split** | Evaluate on unseen future data — never shuffle a time series |
| **Normalization** | Stable gradients; computed from training data *only* to prevent leakage |
| **Log-transform target** | Streamflow is right-skewed; log-space improves learning of low flows |
| **Sequence construction** | LSTMs operate on windows of past observations |

### Data leakage

Computing normalization statistics on the full dataset (including val/test) is a common mistake — the model indirectly "sees" future data. Always fit scalers on **training data only**, then apply the same statistics to val/test.


In [ ]:
# 2.1: Temporal train / val / test split

TRAIN_END = "1999-09-30" # ~19 years of training
VAL_END = "2008-09-30" # ~9 years of validation
# Test period: 2008-10-01 -> 2014-09-30 (~6 years)

dates_idx = pd.DatetimeIndex(loader.dates)
train_mask = dates_idx <= pd.Timestamp(TRAIN_END)
val_mask = (dates_idx > pd.Timestamp(TRAIN_END)) & (dates_idx <= pd.Timestamp(VAL_END))
test_mask = dates_idx > pd.Timestamp(VAL_END)

x_train = loader.forcings[train_mask]
x_val = loader.forcings[val_mask]
x_test = loader.forcings[test_mask]

y_train = loader.target[train_mask]
y_val = loader.target[val_mask]
y_test = loader.target[test_mask]

dates_train = loader.dates[train_mask]
dates_val = loader.dates[val_mask]
dates_test = loader.dates[test_mask]

print(f"Train: {dates_train[0].date()} -> {dates_train[-1].date()}  ({train_mask.sum():,} days)")
print(f"Val:   {dates_val[0].date()} -> {dates_val[-1].date()}    ({val_mask.sum():,} days)")
print(f"Test:  {dates_test[0].date()} -> {dates_test[-1].date()}    ({test_mask.sum():,} days)")


In [ ]:
# 2.2: Normalize forcing variables (z-score)
# Mean and std are computed from TRAINING DATA ONLY.

x_mean = x_train.mean(axis=(0, 1), keepdims=True)  # (1, 1, 6)
x_std = x_train.std(axis=(0, 1), keepdims=True) + 1e-8

x_train_norm = (x_train - x_mean) / x_std
x_val_norm = (x_val - x_mean) / x_std
x_test_norm = (x_test - x_mean) / x_std

print("Normalization statistics (training set only):")
for i, name in enumerate(FORCING_NAMES):
    print(f"  {name}: mean={x_mean[0,0,i]:.3f}, std={x_std[0,0,i]:.3f}")


In [ ]:
# 2.3: Normalize streamflow (log1p + z-score)
# Streamflow is heavily right-skewed — log1p compresses rare flood events
# and prevents the model from focusing only on peak flows.

y_train_log = np.log1p(np.clip(y_train, 0, None))
y_log_mean = float(np.nanmean(y_train_log))
y_log_std = float(np.nanstd(y_train_log[~np.isnan(y_train_log)])) + 1e-8


def normalize_target(y: np.ndarray) -> np.ndarray:
    return (np.log1p(np.clip(y, 0, None)) - y_log_mean) / y_log_std


def denormalize_target(y_norm: np.ndarray) -> np.ndarray:
    return np.expm1(y_norm * y_log_std + y_log_mean)


y_train_norm = normalize_target(y_train)
y_val_norm = normalize_target(y_val)
y_test_norm = normalize_target(y_test)

print(f"NaN values in training target: {np.isnan(y_train).sum():,}  (will be masked during loss computation)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
raw_vals = y_train[~np.isnan(y_train)].ravel()
norm_vals = y_train_norm[~np.isnan(y_train_norm)].ravel()

axes[0].hist(raw_vals, bins=80, color="navy", alpha=0.75, edgecolor="none")
axes[0].set_title("Raw streamflow (ft³/s)")
axes[0].set_xlabel("ft³/s")

axes[1].hist(norm_vals, bins=80, color="steelblue", alpha=0.75, edgecolor="none")
axes[1].set_title("Normalized (log + z-score)")
axes[1].set_xlabel("σ units")

plt.tight_layout()
plt.show()


In [ ]:
# 2.4: Sliding-window Dataset

class StreamflowDataset(Dataset):
    """
    Sliding-window dataset for sequence-to-sequence streamflow prediction.

    Each sample:
        x : (seq_len, n_features) — normalized climate forcings
        y : (seq_len,) — normalized streamflow (NaNs preserved for masking)

    Parameters
    ----------
    x : (time, basins, features) normalized forcings
    y : (time, basins, 1) normalized streamflow
    seq_len : window length in days
    stride : days between consecutive windows (larger -> fewer, faster samples)
    """

    def __init__(self, x: np.ndarray, y: np.ndarray,
                 seq_len: int = 365, stride: int = 1):
        self.seq_len = seq_len
        self.samples = []

        n_time, n_basins, _ = x.shape
        for basin in range(n_basins):
            for t in range(0, n_time - seq_len, stride):
                self.samples.append((
                    x[t:t + seq_len, basin, :].astype(np.float32),
                    y[t:t + seq_len, basin, 0].astype(np.float32),
                ))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.from_numpy(y)

SEQ_LEN = 365  # 1-year context window
STRIDE = 7  # weekly stride balances data coverage and training speed

train_ds = StreamflowDataset(x_train_norm, y_train_norm, seq_len=SEQ_LEN, stride=STRIDE)
val_ds = StreamflowDataset(x_val_norm, y_val_norm, seq_len=SEQ_LEN, stride=STRIDE)

print(f"Training samples: {len(train_ds):,}")
print(f"Validation samples: {len(val_ds):,}")
print(f"Sample shapes — x: {train_ds[0][0].shape}, y: {train_ds[0][1].shape}")


In [ ]:
# 2.5: DataLoaders

BATCH_SIZE = 128

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

x_b, y_b = next(iter(train_loader))
print(f"Mini-batch shapes:")
print(f"  x: {tuple(x_b.shape)} (batch, seq_len, features)")
print(f"  y: {tuple(y_b.shape)} (batch, seq_len)")
print(f"\nAny NaN in x? {torch.isnan(x_b).any().item()}")
print(f"Any NaN in y? {torch.isnan(y_b).any().item()}  <- NaNs in targets are expected")



---
## 3. The LSTM Model

### Why an LSTM?

Streamflow prediction is a classic **long-range dependency** problem. Today's streamflow can depend on precipitation from *months* ago (e.g., snowpack). Standard RNNs suffer from **vanishing gradients** and cannot learn these dependencies.

An **LSTM** adds three *gates* that regulate how information flows through time:

```
  Forget gate : decides what to erase from the cell state (long-term memory)
  Input gate : decides what new information to write into the cell state
  Output gate : decides what part of the cell state to expose as hidden state
```

### Architecture

```
x  (batch, seq_len, 6)
 │
 ▼  Linear encoder  ->  ReLU
   (batch, seq_len, hidden_size)
 │
 ▼  LSTM  (hidden_size -> hidden_size)
   (batch, seq_len, hidden_size)
 │
 ▼  Dropout
 │
 ▼  Linear decoder
   (batch, seq_len, 1)  ->  squeeze  ->  (batch, seq_len)
```

The key **capacity hyperparameter** is `hidden_size`. A larger value gives the model more "working memory" at the cost of more parameters and a higher risk of overfitting.


In [ ]:
class LstmModel(nn.Module):
    """LSTM for sequence-to-sequence streamflow prediction.

    Parameters
    ----------
    n_features : number of input forcing variables
    hidden_size : number of LSTM hidden units (controls model capacity)
    n_layers : number of stacked LSTM layers (controls depth)
    dropout : dropout probability (0 = no regularization)
    """

    def __init__(self, n_features: int, hidden_size: int = 64,
                 n_layers: int = 1, dropout: float = 0.0):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers

        self.input_proj = nn.Linear(n_features, hidden_size)
        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0.0,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Parameters
        ----------
        x : (batch, seq_len, n_features)

        Returns
        -------
        (batch, seq_len)
        """
        x = torch.relu(self.input_proj(x))
        lstm_out, _ = self.lstm(x)
        out = self.output_proj(self.dropout(lstm_out))
        return out.squeeze(-1)


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


N_FEATURES = len(FORCING_NAMES)  # 6
model = LstmModel(n_features=N_FEATURES, hidden_size=64, n_layers=1, dropout=0.4).to(device)

print(model)
print(f"\nTrainable parameters: {count_params(model):,}")

x_dummy = torch.randn(8, SEQ_LEN, N_FEATURES).to(device)
with torch.no_grad():
    out_dummy = model(x_dummy)
print(f"\nForward pass: {tuple(x_dummy.shape)} → {tuple(out_dummy.shape)}  ✓")


> **Exercise** — Change `hidden_size` to 8, 32, and 256. How does the parameter
> count change? (Hint: it scales roughly as `4 x hidden_size^2` due to the LSTM weight matrices.)
>
> **Exercise** — Set `n_layers=2`. What changes in the model printout?



---
## 4. Training the Model

Training iterates over the data in mini-batches:

1. **Forward pass** — input sequences -> predicted streamflow
2. **Loss** — measure prediction error on non-missing timesteps
3. **Backward pass** — `loss.backward()` computes gradients for every parameter
4. **Optimizer step** — nudge weights in the direction that reduces loss

We use **Adam** (adaptive learning rates per parameter) and a **learning-rate scheduler** that halves the LR when validation loss plateaus.

### Loss function

We minimize **MSE in normalized log-space**. NaN positions (missing observations) are masked out so they do not contribute to the gradient.

### Nash-Sutcliffe Efficiency (NSE)

$$\text{NSE} = 1 - \frac{\sum_t (Q_t^\text{obs} - Q_t^\text{sim})^2}
                           {\sum_t (Q_t^\text{obs} - \bar{Q}^\text{obs})^2}$$

| NSE | Meaning |
|-----|---------|
| 1.0 | Perfect prediction |
| 0.0 | No better than predicting the long-term mean every day |
| < 0 | Worse than the mean |

NSE > 0.6 is generally acceptable; NSE > 0.8 is excellent for daily streamflow.


In [ ]:
def masked_mse_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """MSE loss that ignores NaN positions in the target."""
    mask = ~torch.isnan(target)
    if mask.sum() == 0:
        return pred.sum() * 0.0
    return ((pred[mask] - target[mask]) ** 2).mean()


def nse_score(pred: np.ndarray, obs: np.ndarray) -> float:
    """Nash-Sutcliffe Efficiency on raw (denormalized) streamflow arrays."""
    mask = ~np.isnan(obs) & ~np.isnan(pred)
    if mask.sum() < 2:
        return float("nan")
    p, o = pred[mask], obs[mask]
    denom = np.sum((o - np.mean(o)) ** 2)
    return float(1 - np.sum((p - o) ** 2) / denom) if denom > 0 else float("nan")


In [ ]:
def train_model(model, train_loader, val_loader,
                n_epochs=30, lr=1e-3, verbose=True):
    """Train model and return (train_losses, val_losses) per epoch.
    Restores the best (lowest val loss) weights at the end.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=5, factor=0.5
    )
    train_losses, val_losses = [], []
    best_val = float("inf")
    best_state = None

    for epoch in range(n_epochs):
        model.train()
        running = 0.0
        for x_b, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            loss = masked_mse_loss(model(x_b), y_b)  # forward + loss
            loss.backward()  # backprop
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()  # update weights
            running += loss.item()
        train_loss = running / len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_b, y_b in val_loader:
                x_b, y_b = x_b.to(device), y_b.to(device)
                val_loss += masked_mse_loss(model(x_b), y_b).item()
        val_loss /= len(val_loader)

        scheduler.step(val_loss)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if verbose and (epoch + 1) % 5 == 0:
            lr_now = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch+1:3d}/{n_epochs}  "
                  f"train={train_loss:.4f}  val={val_loss:.4f}  lr={lr_now:.2e}")

    model.load_state_dict(best_state)
    return train_losses, val_losses


In [ ]:
model = LstmModel(n_features=N_FEATURES, hidden_size=64, n_layers=1, dropout=0.4).to(device)
print(f"Baseline model — {count_params(model):,} trainable parameters")
print("Training for 30 epochs ...\n")

train_losses, val_losses = train_model(model, train_loader, val_loader, n_epochs=30)
print(f"\nBest val loss: {min(val_losses):.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
epochs = range(1, len(train_losses) + 1)

ax.plot(epochs, train_losses, color="steelblue", linewidth=2, label="Train loss")
ax.plot(epochs, val_losses, color="darkorange", linewidth=2, linestyle="--", label="Val loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE loss  (normalized log-space)")
ax.set_title("Learning Curves — Baseline LSTM")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# What to look for:
# ✓  Both curves decrease and level off -> healthy training
# ✗  Val curve rises while train falls -> overfitting
# ✗  Both curves stay high and flat -> underfitting


---
## 5. Evaluating the Model

After training, we assess performance on the **held-out test period (2008–2014)**.

For a fair evaluation:
- Use the same normalization statistics computed on the training set
- Denormalize predictions back to ft^3/s before computing NSE
- Inspect both aggregate metrics *and* individual hydrographs


In [ ]:
def predict_full_timeseries(model, x_norm: np.ndarray, seq_len: int = 365) -> np.ndarray:
    """Sliding-window inference over an entire time series.

    For each day t >= seq_len, feeds x[t-seq_len:t] into the model and records
    the prediction at day t (the last output of the sequence).

    Returns an array of shape (time, n_basins).
    """
    model.eval()
    n_time, n_basins, _ = x_norm.shape
    preds = np.full((n_time, n_basins), np.nan, dtype=np.float32)

    with torch.no_grad():
        for t in range(seq_len, n_time):
            window = x_norm[t - seq_len:t, :, :].transpose(1, 0, 2)
            x_t = torch.from_numpy(window).float().to(device)
            out = model(x_t)
            preds[t] = out[:, -1].cpu().numpy()

    return preds


pred_norm_test = predict_full_timeseries(model, x_test_norm, seq_len=SEQ_LEN)
pred_cfs_test = denormalize_target(pred_norm_test)
obs_cfs_test = loader.target[test_mask, :, 0]

print("NSE per basin — test period (2008–2014):")
nse_baseline = {}
for i, gid in enumerate(loader.gage_ids):
    nse = nse_score(pred_cfs_test[:, i], obs_cfs_test[:, i])
    nse_baseline[gid] = nse
    print(f"  Basin {gid}: NSE = {nse:.3f}")

print(f"\n  Median NSE: {np.nanmedian(list(nse_baseline.values())):.3f}")
print(f"  Mean NSE:   {np.nanmean(list(nse_baseline.values())):.3f}")


In [ ]:
# Hydrograph — observed vs. predicted
# Try changing basin_idx (0–9)

basin_idx = 0

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

ax = axes[0]
ax.plot(dates_test, obs_cfs_test[:, basin_idx], color="navy", lw=0.8, label="Observed")
ax.plot(dates_test, pred_cfs_test[:, basin_idx], color="tomato", lw=0.8, label="Predicted", alpha=0.85)
ax.set_title(f"Full test period — Basin {loader.gage_ids[basin_idx]}"
             f"   NSE = {nse_baseline[loader.gage_ids[basin_idx]]:.3f}")
ax.set_ylabel("Streamflow (ft^3/s)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

zoom = slice(-548, None)
ax = axes[1]
ax.plot(dates_test[zoom], obs_cfs_test[zoom, basin_idx], color="navy", lw=1.0, label="Observed")
ax.plot(dates_test[zoom], pred_cfs_test[zoom, basin_idx], color="tomato", lw=1.0, label="Predicted", alpha=0.85)
ax.set_title("Zoom: last 18 months")
ax.set_ylabel("Streamflow (ft^3/s)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


In [ ]:
# Observed vs. predicted scatter — one panel per basin
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, (ax, gid) in enumerate(zip(axes.ravel(), loader.gage_ids)):
    obs = obs_cfs_test[:, i]
    pred = pred_cfs_test[:, i]
    mask = ~np.isnan(obs) & ~np.isnan(pred)

    q99 = float(np.nanpercentile(obs, 99))
    ax.scatter(obs[mask], pred[mask], alpha=0.08, s=2, color="steelblue", rasterized=True)
    ax.plot([0, q99], [0, q99], "r--", lw=1)
    ax.set_xlim(0, q99)
    ax.set_ylim(0, q99)
    ax.set_title(f"{gid}\nNSE={nse_baseline[gid]:.2f}", fontsize=9)
    if i >= 5:
        ax.set_xlabel("Observed (ft³/s)", fontsize=8)
    if i % 5 == 0:
        ax.set_ylabel("Predicted (ft³/s)", fontsize=8)
    ax.grid(alpha=0.2)

fig.suptitle("Observed vs. Predicted Streamflow — Test Period", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()



---
## 6. Overfitting and Underfitting

The two most common failure modes in ML:

| | **Underfitting** | **Overfitting** |
|---|---|---|
| **Symptoms** | High loss on both train and val | Low train loss, high val/test loss |
| **Root cause** | Model too simple or training too short | Model too complex, too little data, no regularization |
| **Fix** | Increase capacity, train longer | Add dropout, reduce size, more data, early stopping |

The goal is to find the *sweet spot* where both train and val loss are low and similar.

We run three experiments:

1. **Underfit** — `hidden_size=4`, only 3 epochs
2. **Baseline** — `hidden_size=64`, 30 epochs, dropout=0.4 (already trained above)
3. **Overfit** — `hidden_size=256`, 2 layers, no dropout, trained on just **1 year** of data


In [ ]:
# Experiment A: Underfitting

print("=== Underfitting (hidden=4, 3 epochs) ===")
model_under = LstmModel(n_features=N_FEATURES, hidden_size=4, n_layers=1, dropout=0.0).to(device)
print(f"  Parameters: {count_params(model_under):,}")

tl_under, vl_under = train_model(
    model_under, train_loader, val_loader, n_epochs=3, lr=1e-3, verbose=False
)
pred_under = denormalize_target(predict_full_timeseries(model_under, x_test_norm, SEQ_LEN))
nse_under = np.nanmean([nse_score(pred_under[:, i], obs_cfs_test[:, i])
                        for i in range(loader.n_basins)])
print(f"  Mean NSE (test): {nse_under:.3f}\n")


# Experiment B: Overfitting
# Train on only ONE year of data with a large model and no regularization.

print("=== Overfitting (hidden=256, 2 layers, 1yr data, no dropout) ===")
one_yr = (dates_idx > pd.Timestamp("1990-10-01")) & (dates_idx <= pd.Timestamp("1991-09-30"))
x_tiny_norm = (loader.forcings[one_yr] - x_mean) / x_std
y_tiny_norm = normalize_target(loader.target[one_yr])

tiny_ds = StreamflowDataset(x_tiny_norm, y_tiny_norm, seq_len=SEQ_LEN, stride=1)
tiny_loader = DataLoader(tiny_ds, batch_size=32, shuffle=True) if len(tiny_ds) > 0 else train_loader

model_over = LstmModel(n_features=N_FEATURES, hidden_size=256, n_layers=2, dropout=0.0).to(device)
print(f"  Parameters: {count_params(model_over):,}")

tl_over, vl_over = train_model(
    model_over, tiny_loader, val_loader, n_epochs=60, lr=1e-3, verbose=False
)
pred_over = denormalize_target(predict_full_timeseries(model_over, x_test_norm, SEQ_LEN))
nse_over = np.nanmean([nse_score(pred_over[:, i], obs_cfs_test[:, i])
                       for i in range(loader.n_basins)])
print(f"  Mean NSE (test): {nse_over:.3f}\n")


nse_base_mean = np.nanmean(list(nse_baseline.values()))
print("=== Summary ===")
print(f"  Underfitting: {nse_under:.3f}")
print(f"  Baseline: {nse_base_mean:.3f}")
print(f"  Overfitting: {nse_over:.3f}")


In [ ]:
# Visualize: learning curves (top row) + hydrographs (bottom row)

configs = [
    ("Underfitting\nhidden=4, 3 epochs", tl_under, vl_under, pred_under, nse_under, "#e74c3c"),
    ("Baseline\nhidden=64, dropout=0.4", train_losses, val_losses, pred_cfs_test, nse_base_mean, "#27ae60"),
    ("Overfitting\nhidden=256, 1yr data", tl_over, vl_over, pred_over, nse_over, "#e67e22"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
n_plot = 365
d_plot = dates_test[:n_plot]

for col, (title, tl, vl, preds, nse, c) in enumerate(configs):
    ax = axes[0, col]
    ax.plot(tl, color=c, lw=2, label="Train")
    ax.plot(vl, color="gray", lw=2, linestyle="--", label="Val")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Epoch")
    if col == 0:
        ax.set_ylabel("MSE Loss")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axes[1, col]
    ax.plot(d_plot, obs_cfs_test[:n_plot, 0], color="navy", lw=1.0, label="Observed", alpha=0.9)
    ax.plot(d_plot, preds[:n_plot, 0], color=c, lw=1.0, label="Predicted", alpha=0.85)
    ax.set_title(f"Mean NSE = {nse:.3f}", fontweight="bold")
    if col == 0:
        ax.set_ylabel("Streamflow (ft^3/s)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.tick_params(axis="x", rotation=30, labelsize=8)

fig.suptitle("Underfitting  <-  Sweet Spot -> Overfitting",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("bias_variance_demo.png", dpi=120, bbox_inches="tight")
plt.show()



---
## Summary

You've built a complete, working ML pipeline for streamflow prediction.

| Step | Accomplished |
|------|-------------|
| **Data** | Loaded 10-basin CAMELS data (12,418 days x 6 forcings x 10 basins) |
| **Preprocessing** | Temporal split, z-score normalization, log-transform, sliding windows |
| **Model** | `LstmModel` with `nn.Module` — encoder -> LSTM -> decoder |
| **Training** | Adam optimizer, masked MSE loss, gradient clipping, LR scheduler |
| **Evaluation** | NSE metric, hydrograph plots, scatter diagnostics |
| **Diagnostics** | Demonstrated underfitting, well-tuned baseline, and overfitting |

---
### Things to try
1. **`hidden_size`** — try 16, 128. How does NSE vs. parameter count trade off?
2. **`dropout`** — remove it (`0.0`). Does the val curve diverge from train?
3. **`SEQ_LEN`** — try `30` (1 month) vs `730` (2 years). Does a longer window help?
4. **`STRIDE`** — reduce to `1` for the full dataset. Does more data improve NSE?

---
**Continue to `02_model_augmentation.ipynb`**: diagnose what the model gets wrong, then augment the architecture to improve it.
